### Data Preparation & EDA

##### 1. Setup: Import Library dan Konfigurasi

In [5]:
import sys, os
sys.path.append("..")   # agar config.py di folder induk bisa diimpor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from config import RANDOM_STATE, TEST_SIZE, DATA_PATH, OUTPUT_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print("Python  :", sys.version.split()[0])
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("sklearn :", sklearn.__version__)

Python  : 3.12.10
pandas  : 3.1.1
numpy   : 2.5.5
sklearn : 1.9.1


##### 2. Load Dataset dan Verifikasi
**Deskripsi singkat dataset**
- Setiap baris adalah satu sesi kunjungan ke website e-commerce.
- 17 fitur (aktivitas halaman, bounce/exit rate, page value, bulan, tipe pengunjung, dll.) dan 1 target.
- Target: `Revenue` (True = sesi berakhir dengan pembelian, False = tidak).

In [6]:
df = pd.read_csv(DATA_PATH)
print("Ukuran data:", df.shape)

assert df.shape == (12330, 18), "Ukuran data tidak sesuai, cek file CSV!"
assert "Revenue" in df.columns, "Kolom Revenue tidak ditemukan!"
print("Verifikasi lolos")
df.head()

Ukuran data: (12330, 18)
Verifikasi lolos


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.000,0,0.000,2,64.000,0.000,0.100,0.000,0.000,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.000,0,0.000,2,2.667,0.050,0.140,0.000,0.000,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.000,0,0.000,10,627.500,0.020,0.050,0.000,0.000,Feb,3,3,1,4,Returning_Visitor,True,False


##### 3. Pemeriksaan Struktur dan Tipe Data

Memastikan setiap kolom bertipe sesuai isinya: fitur numerik berupa `int`/`float`, `Month` dan `VisitorType` berupa teks, serta `Weekend` dan `Revenue` berupa boolean.

In [7]:
df.info()
print()
print(df.dtypes.value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType              12330 no

##### 4. Pemeriksaan Missing Value

Missing value dapat menyebabkan error atau bias pada KNN dan Random Forest, sehingga harus diperiksa sebelum modeling.

In [8]:
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Tidak ada missing value")
print("\nTotal missing:", missing.sum())

Tidak ada missing value

Total missing: 0


##### 5. Statistik Deskriptif

Statistik deskriptif digunakan untuk melihat rentang nilai, rata-rata, dan sebaran fitur numerik. Bagian ini juga menunjukkan perbedaan skala antar fitur.

In [9]:
num_cols = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

df[num_cols].describe().T[["min", "mean", "50%", "max"]]

,min,mean,50%,max
Administrative,0.000,2.315,1.000,27.000
Administrative_Duration,0.000,80.819,7.500,"3,398.750"
Informational,0.000,0.504,0.000,24.000
Informational_Duration,0.000,34.472,0.000,"2,549.375"
ProductRelated,0.000,31.731,18.000,705.000
ProductRelated_Duration,0.000,"1,194.746",598.937,"63,973.522"
BounceRates,0.000,0.022,0.003,0.200
ExitRates,0.000,0.043,0.025,0.200
PageValues,0.000,5.889,0.000,361.764
SpecialDay,0.000,0.061,0.000,1.000


##### 6. Pemeriksaan Nilai Nol dan Kolom Kategori Berkode

Banyak fitur aktivitas bernilai 0 karena pengunjung memang tidak membuka jenis halaman tersebut (bukan data hilang). Kolom `OperatingSystems`, `Browser`, `Region`, `TrafficType` berupa kode kategori yang disimpan sebagai angka.

In [10]:
print("Jumlah nilai 0 per fitur numerik:")
print((df[num_cols] == 0).sum())

print("\nJumlah kategori unik pada kolom berkode:")
for col in ["OperatingSystems", "Browser", "Region", "TrafficType"]:
    print(f"{col:18s}: {df[col].nunique()} kategori")

print("\nDistribusi SpecialDay:")
print(df["SpecialDay"].value_counts().sort_index())

Jumlah nilai 0 per fitur numerik:
Administrative              5768
Administrative_Duration     5903
Informational               9699
Informational_Duration      9925
ProductRelated                38
ProductRelated_Duration      755
BounceRates                 5518
ExitRates                     76
PageValues                  9600
SpecialDay                 11079
dtype: int64

Jumlah kategori unik pada kolom berkode:
OperatingSystems  : 8 kategori
Browser           : 13 kategori
Region            : 9 kategori
TrafficType       : 20 kategori

Distribusi SpecialDay:
SpecialDay
0.000    11079
0.200      178
0.400      243
0.600      351
0.800      325
1.000      154
Name: count, dtype: int64


##### 7. Deteksi Data Duplikat

In [11]:
n_dup = df.duplicated().sum()
print("Jumlah baris duplikat:", n_dup, f"({n_dup/len(df)*100:.2f}%)")

print("\nRevenue pada baris duplikat:")
print(df[df.duplicated()]["Revenue"].value_counts())

print("\nContoh baris duplikat:")
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6)

Jumlah baris duplikat: 125 (1.01%)

Revenue pada baris duplikat:
Revenue
False    125
Name: count, dtype: int64

Contoh baris duplikat:


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
8247,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Dec,1,1,1,1,Returning_Visitor,True,False
10751,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Dec,1,1,1,1,Returning_Visitor,True,False
11658,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Dec,1,1,1,1,Returning_Visitor,True,False
8882,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Dec,1,1,1,2,New_Visitor,False,False
11934,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Dec,1,1,1,2,New_Visitor,False,False
11110,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Dec,1,1,1,3,Returning_Visitor,False,False


##### 8. Pemeriksaan Konsistensi Data

Pemeriksaan logika sederhana untuk menemukan nilai yang tidak konsisten.

In [12]:
dur_cols = ["Administrative_Duration", "Informational_Duration", "ProductRelated_Duration"]
print("Nilai negatif pada durasi:", (df[dur_cols] < 0).sum().sum())

total_pages = df["Administrative"] + df["Informational"] + df["ProductRelated"]
print("Sesi dengan 0 halaman dibuka:", (total_pages == 0).sum())

print("BounceRates > ExitRates:", (df["BounceRates"] > df["ExitRates"]).sum())
print("Administrative > 0 tetapi durasi = 0:",
      ((df["Administrative"] > 0) & (df["Administrative_Duration"] == 0)).sum())

Nilai negatif pada durasi: 0
Sesi dengan 0 halaman dibuka: 6
BounceRates > ExitRates: 175
Administrative > 0 tetapi durasi = 0: 135
